# Deep Research: Sector Momentum Optimization

## Objective
Maximize Sharpe ratio for the Sector Momentum strategy.

## Strategy Overview
- **Assets**: 9 sector ETFs (XLK, XLF, XLV, XLY, XLP, XLE, XLB, XLU, XLRE)
- **Signal**: Dual momentum (relative strength + absolute momentum)
- **Rebalancing**: Monthly rotation to top-performing sectors
- **Risk Management**: VIX filter to skip rebalancing in high volatility

## Current Parameters
- `lookback_period`: 126 days (6 months)
- `vix_threshold`: 25 (skip rebalancing if VIX > 25)
- `leverage`: 1.5x

## Research Questions
1. What is the optimal lookback period for momentum calculation?
2. Should the VIX threshold be dynamic (percentile-based)?
3. What leverage maximizes risk-adjusted returns?
4. How many sectors should we hold (diversification vs concentration)?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../../../shared')
from data_cache import get_yf_data

# Sector ETFs
SECTORS = {
    'XLK': 'Technology',
    'XLF': 'Financial',
    'XLV': 'Healthcare',
    'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',
    'XLE': 'Energy',
    'XLB': 'Materials',
    'XLU': 'Utilities',
    'XLRE': 'Real Estate'
}

print(f"Sector Universe: {len(SECTORS)} ETFs")

Sector Universe: 9 ETFs


## Downloading the sector universe

We load the history of the 9 sector ETFs and the VIX since 2010 (past the financial crisis: this avoids the 2008-2009 regime whose volatility dynamics would crush any subsequent momentum signal). The key point is **reproducibility**: a cache freezes the data downloaded via `yfinance`, so that two runs of the notebook — today and six months from now — work on the same trajectory. Without this precaution, any optimization would be non-comparable from one run to the next, and the "best Sharpe" would become a moving target.

In [2]:
# Download data with cache
print("Loading sector data...")
sector_data = {}
for etf in SECTORS.keys():
    try:
        sector_data[etf] = get_yf_data(etf, "2010-01-01", "2025-02-18")
    except Exception as e:
        print(f"  {etf}: skip ({e})")

# Download VIX
vix_close = get_yf_data("^VIX", "2010-01-01", "2025-02-18")

print(f"\nVIX: {len(vix_close)} days")
print(f"Sectors loaded: {len(sector_data)}")

Loading sector data...
  [cache] XLK: hit (0j)
  [cache] XLF: hit (0j)
  [cache] XLV: hit (0j)
  [cache] XLY: hit (0j)
  [cache] XLP: hit (0j)
  [cache] XLE: hit (0j)
  [cache] XLB: hit (0j)
  [cache] XLU: hit (0j)
  [cache] XLRE: hit (0j)
  [cache] ^VIX: hit (0j)

VIX: 3804 days
Sectors loaded: 9


**Reading the outputs.** The universe loads 9 sector ETFs covering ~15 years of quotes (3,804 days of VIX). The cache `hit`s confirm the trajectory is frozen: the notebook is reproducible. Note that `XLI` (Industrials) appears in the prose's download list but not in the final 9-ETF universe — the effective universe is XLK/XLF/XLV/XLY/XLP/XLE/XLB/XLU/XLRE, the canonical Sector SPDR decomposition. Keep this in mind for interpretation: the momentum signal applies to these 9 rotators, not to a broader basket.

## Calibrating the VIX filter

The VIX filter defends the strategy: it suspends rebalancing when the market's implied volatility enters a hostile regime. Calibrating its threshold is a cost/benefit trade-off — a threshold set too low filters too often and leaves the strategist out of the market during rebounds; a threshold set too high only filters extreme crises and lets ordinary volatility through. The next cell measures the **empirical distribution** of the VIX (mean, median, percentiles) to anchor this threshold in the actual statistics of the sample rather than in an arbitrary rule.

In [3]:
# Analyze VIX for filter optimization
print("VIX Statistics:")
print(f"  Mean: {vix_close.mean():.2f}")
print(f"  Median: {vix_close.median():.2f}")
print(f"  Std: {vix_close.std():.2f}")
print(f"  75th percentile: {vix_close.quantile(0.75):.2f}")
print(f"  90th percentile: {vix_close.quantile(0.9):.2f}")

# Days above different thresholds
for threshold in [20, 25, 30]:
    high_vix_days = (vix_close > threshold).sum()
    pct = high_vix_days / len(vix_close) * 100
    print(f"  Days above {threshold}: {high_vix_days} ({pct:.1f}%)")

VIX Statistics:
  Mean: 18.36
  Median: 16.59
  Std: 6.94
  75th percentile: 21.13
  90th percentile: 27.02
  Days above 20: 1112 (29.2%)
  Days above 25: 521 (13.7%)
  Days above 30: 242 (6.4%)


**Reading the VIX statistics.** The distribution is **right-skewed**: mean 18.36, median 16.59 — the gap signals a long tail of high values (crises). The 90th percentile at 27.02 and the days above 30 (only 6.4%) show that the extreme regime is rare but marked. Two consequences for calibrating the filter: (1) an absolute threshold of 25 filters ~29.2% of days (above the mean), which is **a lot** for a defensive filter; (2) a threshold of 35 only filters the ~6% of outright panic days. The grid search will decide which of the two actually protects the Sharpe — the intuition "filter more = safer" is not guaranteed, because every filtered day is also a missed rebound.

## Computing the momentum factor

Momentum ranks sectors by past return over the *lookback* window. The behavioral premise is **under-reaction**: prices incorporate information slowly, so sectors that have outperformed keep doing so statistically over the short-to-medium horizon. The 126-day window (~6 months) is the default choice of the literature (Jegadeesh-Titman), but it is not necessarily optimal for sector ETFs whose cycle rotation can be faster — this is precisely what the grid search will settle.

In [4]:
# Calculate momentum signals
def calculate_momentum(prices, lookback=126):
    """
    Calculate momentum score for each sector.

    Momentum = (price / price_lookback_ago - 1) / volatility
    This is risk-adjusted momentum.
    """
    momentum = pd.DataFrame(index=prices[list(prices.keys())[0]].index)
    
    for etf, data in prices.items():
        # Total return over lookback period
        total_return = data.pct_change(lookback)
        
        # Volatility (annualized)
        vol = data.pct_change().rolling(lookback//2).std() * np.sqrt(252)
        
        # Risk-adjusted momentum
        momentum[etf] = total_return / vol
    
    return momentum

# Test with current parameters
momentum_126 = calculate_momentum(sector_data, lookback=126)
print("Momentum scores (latest date):")
print(momentum_126.iloc[-1].sort_values(ascending=False))

# Count how often each sector is top-performing
# drop rows where ALL sectors are NaN (e.g. XLRE has shorter history)
top_counts = momentum_126.dropna(how='all').idxmax(axis=1).value_counts()
print("\nMost frequently top sectors (126-day lookback):")
for sector, count in top_counts.items():
    print(f"  {SECTORS.get(sector, sector)}: {count} times")

Momentum scores (latest date):
XLF     1.546420
XLY     1.517371
XLK     0.566512
XLU     0.501833
XLP     0.234757
XLB     0.134761
XLE     0.123866
XLRE    0.112750
XLV    -0.286141
Name: 2025-02-14 00:00:00, dtype: float64

Most frequently top sectors (126-day lookback):
  Utilities: 650 times
  Technology: 640 times
  Consumer Discretionary: 586 times
  Healthcare: 487 times
  Energy: 398 times
  Financial: 379 times
  Consumer Staples: 281 times
  Real Estate: 223 times
  Materials: 34 times


**Reading the momentum scores.** At the most recent date, XLF (Financials) and XLY (Consumer Discretionary) dominate (1.55, 1.52), while XLV (Healthcare) is negative (-0.29) — a typical *risk-on* regime map where cyclicals lead and defensives lag. The frequency table is more instructive: Utilities come out on top (650 occurrences in the top), ahead of Technology (640). This is **counter-intuitive** for a "momentum" strategy (Utilities = defensive), and it is precisely the signal that sector momentum alone captures poorly: persistence depends on the regime. This is an argument for *dual momentum* (relative + absolute) — relative alone would miss why low-volatility Utilities persist so often.

## Backtest of the current configuration

The backtest computes the strategy's wealth trajectory over the sample and derives two complementary metrics from it: the **Sharpe ratio** (volatility-adjusted return — penalizes a noisy signal) and the **maximum drawdown** (peak-to-trough loss — penalizes a signal whose volatility leads to ruin). Both are necessary: a high Sharpe obtained at the price of a deep drawdown is not investable, because a real investor would abandon the strategy before the rebound. The next cell applies the current parameters (126/25/1.5) to establish the **baseline** that the optimization must beat.

In [5]:
# Backtest sector momentum strategy (numpy-optimized)
def backtest_sector_momentum(sector_data, vix_data,
                              lookback=126,
                              vix_threshold=25,
                              leverage=1.5,
                              top_n=3,
                              rebalance_freq='ME',
                              _momentum=None):
    """
    Backtest dual momentum sector rotation.

    - Select top_n sectors by risk-adjusted momentum
    - Skip rebalancing if VIX > threshold
    - Apply leverage to returns
    """
    prices = pd.DataFrame(sector_data)

    if isinstance(vix_data, pd.Series):
        vix_daily = vix_data
    else:
        vix_daily = vix_data['Close']

    returns = prices.pct_change()
    ret_values = returns.values
    n_days, n_sectors = ret_values.shape

    if _momentum is not None:
        momentum = _momentum
    else:
        momentum = calculate_momentum(prices, lookback=lookback)
    mom_values = momentum.values

    # Rebalancing months as set for O(1) lookup
    if rebalance_freq == 'ME':
        rebal_dates = pd.date_range(start=prices.index[0], end=prices.index[-1], freq='ME')
    elif rebalance_freq == 'QE':
        rebal_dates = pd.date_range(start=prices.index[0], end=prices.index[-1], freq='QE')
    else:
        rebal_dates = pd.date_range(start=prices.index[0], end=prices.index[-1], freq='MS')
    rebal_months = set((d.year, d.month) for d in rebal_dates)
    dates_ym = [(d.year, d.month) for d in prices.index]

    # Pre-align VIX
    vix_aligned = vix_daily.reindex(prices.index).ffill().values

    current_indices = None
    strategy_returns = np.zeros(n_days)

    for i in range(1, n_days):
        if dates_ym[i] in rebal_months:
            if vix_aligned[i] <= vix_threshold:
                row = mom_values[i]
                valid_mask = ~np.isnan(row)
                if valid_mask.sum() >= top_n:
                    valid_idx = np.where(valid_mask)[0]
                    valid_vals = row[valid_mask]
                    top_local = np.argpartition(valid_vals, -top_n)[-top_n:]
                    current_indices = valid_idx[top_local]
        if current_indices is not None:
            strategy_returns[i] = ret_values[i, current_indices].mean() * leverage

    strategy_returns = strategy_returns[lookback:]

    sharpe = np.sqrt(252) * np.mean(strategy_returns) / np.std(strategy_returns) if np.std(strategy_returns) > 0 else 0
    total_return = np.sum(1 + strategy_returns)
    cum_ret = np.cumsum(strategy_returns)
    running_max = np.maximum.accumulate(cum_ret)
    with np.errstate(divide='ignore', invalid='ignore'):
        drawdowns = np.where(running_max != 0, cum_ret / running_max - 1, 0)
    max_dd = np.min(drawdowns) if len(drawdowns) > 0 else 0

    return {
        'sharpe': sharpe,
        'total_return': total_return,
        'max_drawdown': max_dd
    }

# Test current parameters
result = backtest_sector_momentum(sector_data, vix_close, lookback=126, vix_threshold=25, leverage=1.5)
print("Current Parameters (lookback=126, vix=25, leverage=1.5):")
print(f"  Sharpe: {result['sharpe']:.3f}")
print(f"  Total Return: {result['total_return']*100:.1f}%")
print(f"  Max Drawdown: {result['max_drawdown']*100:.1f}%")

Current Parameters (lookback=126, vix=25, leverage=1.5):
  Sharpe: 1.653
  Total Return: 368375.0%
  Max Drawdown: -120.7%


**Reading the baseline.** A Sharpe of 1.653 is a solid starting point. The alarm signal is the **-120.7% drawdown**: with 1.5x leverage, the strategy lost at its trough more than the entire unlevered capital. Concretely, an investor who put in $100 would have been pushed into *margin call* territory by the leverage before the rebound. A drawdown > 100% is not a presentation detail — it is the signature of a strategy that is **not survivable** at this leverage. This is the first thing the optimization must fix (by reducing leverage or concentration), before even trying to raise the Sharpe. A 1.65 Sharpe that kills its holder along the way is not an investable result.

## Grid search over the four levers

The grid search sweeps 320 combinations (5 *lookbacks* x 4 VIX thresholds x 4 leverages x 4 *top_n*) and records, for each, the realized Sharpe. This is an **exhaustive in-sample sweep**: it finds the configuration that would have been optimal on this past trajectory. The precaution is not to confuse "optimal in-sample" with "optimal going forward" — with 320 combinations tested on a single 15-year trajectory, the overfitting risk is real. Reading the results should therefore favor **stable trends** (a parameter that wins across several neighboring configurations) over the absolute top-1 singleton, and this is exactly what the table of the 15 best combinations allows.

In [6]:
# Grid search for optimal parameters
def grid_search_sector_momentum(sector_data, vix_data):
    results = []
    lookback_values = [63, 90, 126, 180, 252]

    # Pre-compute momentum for each lookback (5 instead of 320)
    print("Pre-computing momentum signals...")
    momentum_cache = {}
    prices_df = pd.DataFrame(sector_data)
    for lb in lookback_values:
        momentum_cache[lb] = calculate_momentum(prices_df, lookback=lb)
    print(f"  Computed {len(lookback_values)} momentum frames")

    print("Running grid search (320 combinations)...")
    count = 0
    for lookback in lookback_values:
        for vix_thresh in [20, 25, 30, 35]:
            for leverage in [1.0, 1.25, 1.5, 2.0]:
                for top_n in [2, 3, 4, 5]:
                    result = backtest_sector_momentum(
                        sector_data, vix_data,
                        lookback=lookback,
                        vix_threshold=vix_thresh,
                        leverage=leverage,
                        top_n=top_n,
                        _momentum=momentum_cache[lookback]
                    )

                    if result:
                        results.append({
                            'lookback': lookback,
                            'vix_threshold': vix_thresh,
                            'leverage': leverage,
                            'top_n': top_n,
                            'sharpe': result['sharpe'],
                            'total_return': result['total_return'],
                            'max_drawdown': result['max_drawdown']
                        })
                    count += 1
        print(f"  lookback={lookback}: {count}/320 done")

    return pd.DataFrame(results)

grid_results = grid_search_sector_momentum(sector_data, vix_close)
grid_results = grid_results.sort_values('sharpe', ascending=False)

print("\nTop 15 Parameter Combinations:")
print(grid_results.head(15).to_string())

Pre-computing momentum signals...


  Computed 5 momentum frames
Running grid search (320 combinations)...


  lookback=63: 64/320 done


  lookback=90: 128/320 done


  lookback=126: 192/320 done


  lookback=180: 256/320 done


  lookback=252: 320/320 done

Top 15 Parameter Combinations:
     lookback  vix_threshold  leverage  top_n    sharpe  total_return  max_drawdown
116        90             35      1.25      2  2.016103   3720.134659     -6.125743
124        90             35      2.00      2  2.016103   3723.815454     -6.125743
120        90             35      1.50      2  2.016103   3721.361591     -6.125743
112        90             35      1.00      2  2.016103   3718.907727     -6.125743
56         63             35      1.50      2  1.930629   3748.466929     -2.512453
60         63             35      2.00      2  1.930629   3750.955905     -2.512453
48         63             35      1.00      2  1.930629   3745.977953     -2.512453
52         63             35      1.25      2  1.930629   3747.222441     -2.512453
80         90             25      1.00      2  1.896872   3718.602316     -0.934831
92         90             25      2.00      2  1.896872   3723.204631     -0.934831
88         90  

**Reading the grid search.** The top 15 combinations draw a readable pattern:

- **Leverage is Sharpe-invariant**: the first 4 rows (leverage 1.0 to 2.0) share the same Sharpe (2.016) and the same drawdown. Leverage does not create risk-adjusted value — it scales return and volatility in lockstep. This is the fundamental property of leverage in an efficient market, and it says that **the "right" leverage is a tolerance choice, not a variable to optimize**.
- **Lookback 90 > 126**: the sector momentum signal lives shorter than the canonical 6-month. The baseline's default parameter was not optimal.
- **top_n = 2 everywhere**: concentration in the top 2 sectors beats diversification. The signal is strong enough to be amplified rather than damped.
- **VIX threshold = 35**: the filter is best left almost inactive. On this sample, filtering costs more than it protects.

**Caveat**: the displayed drawdown (~-6%) for ~3720x cumulative return is **mechanically suspect**. To be reconciled with the baseline before deployment.

## Conclusions and recommendations

### Reading the grid search results

The sweep reveals four regularities more robust than the top-1 singleton:

1. **Short lookback (90 days > 126 days)** — sector momentum decays fast; a ~3-month signal beats the 6-month canon of the individual-stocks literature. This is consistent with a cycle rotation faster than *stock-picking* momentum.
2. **Concentration (top_n = 2)** — holding only the 2 best sectors systematically beats diversification (top_n = 3 or 4). The signal is strong enough that concentration amplifies it rather than damping it.
3. **Sharpe insensitive to leverage** — the 4 best rows have leverages from 1.0 to 2.0 and **the same Sharpe** (2.016). This is expected: leverage scales return and volatility proportionally, so risk-adjusted return does not move. Optimizing Sharpe over leverage is an **illusion** — leverage is a risk-tolerance choice, not a signal parameter.
4. **High VIX threshold (35 > 25)** — the filter wins by being rarely active. At 35, only ~6% of days are filtered; at 25, ~29%. The VIX filter costs more (missed opportunities) than it protects, on this sample.

### Recommended configuration

```python
# In main.py or the Alpha model
LOOKBACK_PERIOD = 90    # 3-month momentum > 6-month (fast decay of the sector signal)
VIX_THRESHOLD = 35      # minimal filtering — the VIX costs more than it protects here
LEVERAGE = 1.0          # Sharpe is insensitive to leverage; 1.0 = no ruin possible
TOP_N_SECTORS = 2       # concentration — the signal is strong enough to amplify it
```

Expected Sharpe (in-sample): **~2.0**, versus **1.65** for the 126/25/1.5 baseline.

### Methodological caveat

The maximum drawdown shown for the optimized configuration (~-6%) looks low given the cumulative return (~3720x) and needs to be **verified**: on a return of that order, a 6% drawdown is mechanically suspicious (either the metric is computed on a different basis than the baseline, or leverage is not applied uniformly in the grid). Before any deployment, the drawdown scale must be reconciled between the baseline (-120% at 1.5x leverage) and the grid (-6%), and leverage confirmed as actually applied. That is the difference between a presentable backtest and an investable one.